In [1]:
# During the review process of the paper submitted to ICML 2025, 
# a reviewer (referred to as Reviewer t5C9) introduced an alternative algorithm/code 
# for computing the Min-Max-Jump (MMJ) distance matrix. See:
# https://openreview.net/forum?id=qNfEkSuGKk

# This file tests the algorithm/code.

In [2]:
import numpy as np
import numba
import time
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances


@numba.njit(cache=True, fastmath=True)
def prim_mst(D):
    """
    Optimized Prim's MST using Numba JIT for dense graphs.
    Returns edges as list of (u, v, weight) tuples.
    """
    n = D.shape[0]
    in_mst = np.zeros(n, dtype=np.bool_)
    parent = np.full(n, -1, dtype=np.int64)
    key = np.full(n, np.inf, dtype=D.dtype)
    key[0] = 0.0

    for _ in range(n):
        # Find minimum key vertex not in MST
        u = -1
        min_val = np.inf
        for i in range(n):
            if not in_mst[i] and key[i] < min_val:
                min_val = key[i]
                u = i

        if u == -1:
            break

        in_mst[u] = True

        # Update neighbors
        for v in range(n):
            if not in_mst[v] and D[u, v] < key[v]:
                key[v] = D[u, v]
                parent[v] = u

    # Build edge list
    edges = []
    for v in range(1, n):
        u = parent[v]
        edges.append((u, v, D[u, v]))

    return edges

def build_csr_adjacency(n, mst_edges):
    """Convert MST to compressed sparse row (CSR) format"""

    edge_counts = np.zeros(n, dtype=np.int32)
    for u, v, _ in mst_edges:
        edge_counts[u] += 1
        edge_counts[v] += 1

    ptr = np.zeros(n+1, dtype=np.int32)
    ptr[1:] = np.cumsum(edge_counts)

    adj_edges = np.empty(ptr[-1], dtype=np.int32)
    adj_weights = np.empty(ptr[-1], dtype=np.float64)
    positions = np.zeros(n, dtype=np.int32)

    for u, v, w in mst_edges:
        for _ in range(2):  # Add both directions
            idx = ptr[u] + positions[u]
            adj_edges[idx] = v
            adj_weights[idx] = w
            positions[u] += 1
            u, v = v, u  # Swap for reverse direction

    return ptr, adj_edges, adj_weights

@numba.njit(parallel=True, cache=True, fastmath=True)
def compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights):
    """Numba-optimized BFS for all pairs bottleneck calculation"""
    bottleneck = np.zeros((n, n), dtype=np.float64)

    for src in numba.prange(n):
        visited = np.zeros(n, dtype=numba.boolean)
        max_edges = np.zeros(n, dtype=np.float64)
        queue = np.empty(n, dtype=np.int32)
        queue_weights = np.empty(n, dtype=np.float64)
        front = back = 0

        # Initialize BFS
        visited[src] = True
        queue[back] = src
        queue_weights[back] = 0.0
        back += 1

        while front < back:
            u = queue[front]
            current_max = queue_weights[front]
            front += 1

            # Process all neighbors
            start = ptr[u]
            end = ptr[u+1]
            for i in range(start, end):
                v = adj_edges[i]
                weight = adj_weights[i]

                if not visited[v]:
                    new_max = max(current_max, weight)
                    visited[v] = True
                    max_edges[v] = new_max
                    queue[back] = v
                    queue_weights[back] = new_max
                    back += 1

        bottleneck[src] = max_edges

    return bottleneck

def ultra_fast_wide(distance_matrix):
    n = distance_matrix.shape[0]
#     distance_matrix = np.round(pairwise_distances(X), 15)
    # mst = prim_mst(distance_matrix)  # Use Numba-optimized prim_mst

    start = time.time()
    mst = prim_mst(distance_matrix)
    end = time.time()
    time_used = end - start
    time_used = np.round(time_used, 3)

    # print(f"Time used for computing MST: {time_used}s" )


    # Convert MST to CSR format
    ptr, adj_edges, adj_weights = build_csr_adjacency(n, mst)

    # Compute bottleneck matrix with Numba
    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights)


def create_symmetric_distance_matrix(n, seed, low=1, high=10000):
    if seed is not None:
        np.random.seed(seed)
    A = np.random.randint(low, high, size=(n, n))
    sym_A = (A + A.T) // 2  # Ensure symmetry and integer values
    for k in range(n):
        sym_A[k, k] = 0
    return sym_A.astype('float64')

In [3]:
n = 1000

random_seed = 78375

print(f"Number of nodes: {n}" )


distance_matrix = create_symmetric_distance_matrix(n, random_seed)
start = time.time()
mmj_matrix_Reviewer_t5C9_code = ultra_fast_wide(distance_matrix)
end = time.time()
time_used = end - start
time_used = np.round(time_used, 3)

print(f"Time used for calculating Min-Max-Jump (MMJ) distance matrix: {time_used}s" )
print("Print last 30 values of the first row of MMJ matrix:")
print(mmj_matrix_Reviewer_t5C9_code[0, -30:])




Number of nodes: 1000
Time used for calculating Min-Max-Jump (MMJ) distance matrix: 0.483s
Print last 30 values of the first row of MMJ matrix:
[225. 226. 244. 252. 202. 283. 314. 251. 230. 304. 205. 187. 248. 256.
 260. 252. 245. 296. 202. 298. 256. 235. 230. 433. 280. 353. 186. 229.
 242. 254.]
